In [1]:
import re
import unicodedata
from datetime import datetime
from pyspark.sql import Row
from pyspark.sql.functions import col
from delta.tables import DeltaTable

# Intentar importar langdetect para detección de idioma
try:
    from langdetect import detect
    HAS_LANGDETECT = True
except ImportError:
    HAS_LANGDETECT = False

# ---------------------------------------------------------
# FUNCIONES DE LIMPIEZA Y NORMALIZACIÓN DE TEXTO
# ---------------------------------------------------------

def normalize_characters(text: str) -> str:
    """Normaliza caracteres Unicode (NFC), preservando tildes y caracteres en español."""
    if not text:
        return ""
    normalized = unicodedata.normalize('NFC', text)
    normalized = normalized.replace('\x00', '').replace('“', '"').replace('”', '"').replace('’', "'")
    return normalized

def remove_repetitive_headers_footers(text: str) -> str:
    """Elimina patrones comunes de encabezados y pies de página (páginas numeradas, etc.)."""
    if not text:
        return ""
    pattern_pages = re.compile(r'(?i)(página|pág\.|page)\s+\d+(\s+(de|of)\s+\d+)?', re.IGNORECASE)
    cleaned = pattern_pages.sub('', text)
    
    pattern_dates = re.compile(r'\b\d{2}[/-]\d{2}[/-]\d{4}\b')
    cleaned = pattern_dates.sub('', cleaned)
    
    return cleaned

def clean_and_normalize_pipeline(raw_text: str) -> str:
    """Pipeline completo de limpieza y normalización."""
    if not raw_text:
        return ""
    
    text = normalize_characters(raw_text)
    text = remove_repetitive_headers_footers(text)
    
    # Unir líneas rotas a mitad de frase
    text = re.sub(r'(?<=[a-zA-Záéíóúñ0-9,])\n(?=[a-zA-Záéíóúñ])', ' ', text)
    # Limpiar saltos de línea múltiples
    text = re.sub(r'\n{3,}', '\n\n', text)
    # Eliminar espacios múltiples horizontalmente
    text = re.sub(r'[ \t]{2,}', ' ', text)
    
    return text.strip()

def detect_language(text: str) -> str:
    """Detecta el idioma del texto (código ISO 639-1). Fallback a 'es'."""
    if not text or len(text.strip()) < 20:
        return "es"
    if HAS_LANGDETECT:
        try:
            return detect(text[:1000])
        except Exception:
            return "es"
    return "es"

def calculate_quality_score(clean_text: str, category: str, source_url: str, title: str):
    """
    Calcula el Quality Score (0-100) y determina el estado (valid, warning, invalid).
    """
    score = 100
    num_chars = len(clean_text) if clean_text else 0
    
    if not clean_text or num_chars == 0:
        score -= 30
        
    if num_chars < 500:
        score -= 20
        
    if not category or category in ["unknown", "none", ""]:
        score -= 10
        
    if not source_url or source_url.strip() == "":
        score -= 10
        
    if not title or title.strip() == "":
        score -= 10
        
    if score >= 80:
        status = "valid"
    elif score >= 50:
        status = "warning"
    else:
        status = "invalid"
        
    return float(score), status

# ---------------------------------------------------------
# PROCESAMIENTO Y REFINADO DE LA TABLA SILVER
# ---------------------------------------------------------

# 1. Leer los registros actuales de silver_documents sin prefijo de esquema
raw_silver_df = spark.table("silver_documents").collect()

refined_rows = []

for row in raw_silver_df:
    doc_id = row.document_id
    file_name = row.file_name
    title = row.document_title if hasattr(row, "document_title") and row.document_title else file_name
    category = row.document_category if hasattr(row, "document_category") and row.document_category else "operations"
    
    # Ruta OneLake
    source_url = f"abfss://Ecodocs@onelake.dfs.fabric.microsoft.com/LH_Ecodocs.Lakehouse/Files/Bronze/Documents/{file_name}"
    
    raw_text = row.clean_text if hasattr(row, "clean_text") else ""
    cleaned_text = clean_and_normalize_pipeline(raw_text)
    
    language = detect_language(cleaned_text)
    
    num_characters = len(cleaned_text)
    num_words = len(cleaned_text.split()) if cleaned_text else 0
    
    quality_score, quality_status = calculate_quality_score(
        clean_text=cleaned_text,
        category=category,
        source_url=source_url,
        title=title
    )
    
    refined_rows.append(Row(
        document_id=doc_id,
        file_name=file_name,
        document_title=title,
        document_category=category,
        language=language,
        clean_text=cleaned_text,
        num_pages=row.num_pages if hasattr(row, "num_pages") else 1,
        num_characters=num_characters,
        num_words=num_words,
        source_url=source_url,
        quality_score=quality_score,
        quality_status=quality_status,
        extraction_date=datetime.now()
    ))

# ---------------------------------------------------------
# GUARDAR RESULTADOS REFINADOS
# ---------------------------------------------------------

if refined_rows:
    df_refined = spark.createDataFrame(refined_rows)
    
    # Sobrescribir directamente en la tabla del Lakehouse activo
    df_refined.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver_documents")
    
    print(f" Refinados y actualizados {len(refined_rows)} documentos en 'silver_documents'.")
else:
    print("No se encontraron registros para refinamiento.")

StatementMeta(, 051cdc43-e38e-4057-b236-007b135a9b94, 3, Finished, Available, Finished, False)

 Refinados y actualizados 8 documentos en 'silver_documents'.


In [3]:
# Consulta de validación del dataset refinado en Silver
spark.sql("""
    SELECT 
        file_name, 
        document_category, 
        language, 
        num_characters, 
        quality_score, 
        quality_status
    FROM silver_documents
    ORDER BY quality_score DESC
""").show(truncate=False)

StatementMeta(, 051cdc43-e38e-4057-b236-007b135a9b94, 5, Finished, Available, Finished, False)

+----------------------------------------------------------+-----------------+--------+--------------+-------------+--------------+
|file_name                                                 |document_category|language|num_characters|quality_score|quality_status|
+----------------------------------------------------------+-----------------+--------+--------------+-------------+--------------+
|legal_Reglamento 2016-679.pdf                             |legal            |es      |216129        |100.0        |valid         |
|security_manual_seguridad_plantas_solares.txt             |security         |es      |501           |100.0        |valid         |
|esg_IB_Informe_Sostenibilidad.pdf                         |esg              |es      |485784        |100.0        |valid         |
|technical_Guia_Profesional_Tramitacion_autoconsumo_v.6.pdf|technical        |es      |413237        |100.0        |valid         |
|hr_politica_teletrabajo_ecopower.txt                      |hr              